# Очистка данных кафе

Здесь будет описание проекта.

In [7]:
https://colab.research.google.com/github/peneeeee1/junior-analyst-portfolio/blob/main/cafe-data-cleaning/notebook.ipynb сыллка поменялся формат или из-за того что заполнила файл

SyntaxError: invalid syntax (1632683786.py, line 1)

In [17]:
%pip freeze > requirements.txt
import pandas as pd


# Используем "Raw" ссылку (обязательно слово raw.githubusercontent.com)
url = 'https://raw.githubusercontent.com/peneeeee1/junior-analyst-portfolio/main/cafe-data-cleaning/dirty_cafe_sales.csv'
df = pd.read_csv(url)

# Знакомство с данными (первичное)
print(df.head(20))
print(df.info())
print(df.describe())








   Transaction ID      Item Quantity Price Per Unit Total Spent  \
0     TXN_1961373    Coffee        2            2.0         4.0   
1     TXN_4977031      Cake        4            3.0        12.0   
2     TXN_4271903    Cookie        4            1.0       ERROR   
3     TXN_7034554     Salad        2            5.0        10.0   
4     TXN_3160411    Coffee        2            2.0         4.0   
5     TXN_2602893  Smoothie        5            4.0        20.0   
6     TXN_4433211   UNKNOWN        3            3.0         9.0   
7     TXN_6699534  Sandwich        4            4.0        16.0   
8     TXN_4717867       NaN        5            3.0        15.0   
9     TXN_2064365  Sandwich        5            4.0        20.0   
10    TXN_2548360     Salad        5            5.0        25.0   
11    TXN_3051279  Sandwich        2            4.0         8.0   
12    TXN_7619095  Sandwich        2            4.0         8.0   
13    TXN_9437049    Cookie        5            1.0         5.

In [8]:
#Работа с дубликатами:
#Просмотр через цикл дублей в названии:

for col in df.columns:
    unique_values = df[col].unique()
    print(unique_values)

"""Полных дублей нет так как id полность уник., в названиях нет(была проверка через цикл),частичные найти
проблематично т.к нет мерила их выявления(ключа). Возможно в стобце с id присутствуют строки не id которые
могут скрыть полные дубликаты."""

#Просмотр наличия уник. значений в столбце с id без TXN

df[~df['Transaction ID'].str.contains('TXN')]['Transaction ID'].unique()
#Пусто

"""Осталось проверить на выбросы в  возможных частичных дублях чтобы бы посмотреть где возможно есть дубли,
но пока нет ключа что считать за дубли , удалять их нельзя. """
#Просмотр количества повторов строк
result = (
    df.groupby(list(df.columns[1:8]))
      .size()
      .reset_index(name='колич') # Внимание: name (ед.ч.), так как уровень индекса теперь один
      .sort_values('колич', ascending=False)
)
print(result.boxplot())
print(result[result['колич']>1]['колич'].count())
#66 дней из 365 было две строки  с полным повтором информации о покупке(кроме уник. id)


"""Итого: проделана в меру детальная работа по поиску дублей, но обнаружить их не удалось,
либо руководитель ошибся и дублей в данных нет, либо нужно делать более детальный осмотр для
поиска.Возможно дубли связаны с наличием в таблице одноверменно и 'UNKNOWN' 'ERROR' и nan"""



['TXN_1961373' 'TXN_4977031' 'TXN_4271903' ... 'TXN_5255387' 'TXN_7695629'
 'TXN_6170729']
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']
[2. 4. 5. 3. 1.]
[2.  3.  1.  5.  4.  1.5]
['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' 3.0 nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan 3.0]
['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR' 3.0]
['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28' '2023-07-28' '2023-12-31'
 '2023-11-07' 'ERROR' '2023-05-03' '2023-06-01' '2023-03-21' '2023-11-15'
 '2023-06-10' '2023-02-24' '2023-03-25' '2023-01-15' 3.0 '2023-03-30'
 '2023-12-01' '2023-09-18' '2023-06-03' '2023-12-13' '2023-04-20'
 '2023-04-10' '2023-03-11' '2023-06-02' '2023-11-06' '2023-08-15'
 '2023-10-09' '2023-05-28' '2023-07-17' '2023-04-29' '2023-06-08'
 '2023-06-29' '2023-04-17' '2023-12-22' '2023

TypeError: bad operand type for unary ~: 'float'

In [ ]:
#Поиск ошибок в ценах

"""Возможно менеджер имел в виду ошибка в ценах в несоответствии(наименований товаров 8 а цен возможных
в таблице 6 если только разные виды товаров не имеют одну цену)"""
#Проверка имеют ли некоторые товары одну цену или цена определенных товаров не указана:
result = df[~df['Price Per Unit'].astype(str).str.contains('UNKNOWN|ERROR', na=True)]['Item'].unique()
print(result)

"""Заполнение ценами исходя из данных в таблице"""

#Приводим столбец с ценой к числовому формату, нечисла в Nan
df['Price Per Unit']=pd.to_numeric(df['Price Per Unit'],errors='coerce')

# Создаем маску: где цена указана (не NaN), а где заглушка
has_price = df['Price Per Unit'].notnull()
print(df['Price Per Unit'].isnull().sum())

#  Для каждой группы (Товар) берем первое НЕПУСТОЕ значение цены
# transform ищет первое валидное число внутри каждой группы ('Десерт', 'Напиток'...)

correct_prices = df.groupby('Item', dropna=False)['Price Per Unit'].transform('first')


# 3. Заполняем только те места, где была заглушка/пустота
df.loc[~has_price, 'Price Per Unit'] = correct_prices[~has_price]

"""Итого: вместо 533 строк где была не указана цена, они  заполнены исходя из имеющихся данных, теперь
расчет показателей для отчета получится полнее)"""


print(df.head(20))




#Корректировка форматов столбцов object:

"""(делать вычисления мешают заглушки 'UNKNOWN' и 'ERROR'
для дальнейшей работы они информации не дают поэтому для дальнейших расчетов их нужно  заполнить например
медианой(столбец quantity)."""
#Для выявления медианы нечисловые строки нужно заполнить Nan.:
df['Quantity']=pd.to_numeric(df['Quantity'],errors='coerce')
count_valid=df['Quantity'].count()
print(count_valid)
#Заполняем nan  медианой
df[df['Quantity'].isnull()] = df['Quantity'].median()
print(df.info())


In [28]:
#Обработка оставшихся пропусков в столбцах необходимых для витрины отчёта":
"""В столбце item если заполнить пропуски по недостающим данным из таблицы, возможно получить
менее точный итог чем в таблице с ценами,так как  одна цена может относится к разным
наименованиям товаров,  лучше заполнить пропуски заглушками nan чтобы гарантированее учесть в расчетах,  строки 'ERROR' и
'UNKNOWN' заполнить исходя из данных получится компромис между возможной дезинформации количества
товаров и стоимости"""
has_item = df['Item'].astype(str).str.contains('UNKNOWN|ERROR', na=True)
correct_item = df.groupby('Price Per Unit', dropna=False)['Item'].transform('first')
df.loc[has_item, 'Item'] = correct_item[has_item]
df.head(20)
"""Обнаружилось что цена 1.5 идет только в паре с error,значит в датасете недостает информации по
определенному продукту,строки 'ERROR' в столбце нужно обозначить специальной заглушкой"""

#Nan на 'Nan','error' на 'товар за 1.5'
df['Item']=df['Item'].fillna('Nan')
df[df['Item'] == 'ERROR']='товар за 1.5'
print(df['Item'].head(20))






0           Coffee
1             Cake
2           Cookie
3            Salad
4           Coffee
5         Smoothie
6             Cake
7         Sandwich
8              Nan
9         Sandwich
10           Salad
11        Sandwich
12        Sandwich
13          Cookie
14    товар за 1.5
15           Salad
16        Sandwich
17           Juice
18            Cake
19           Juice
Name: Item, dtype: object
